# v2 validation check — team-archetype grid → LR (polynomial), DRAW market

The team-archetype grid region (see `goals_foragst_team_kmeans_lr_home.ipynb` for the full description), applied to the **draw** market: cluster individual teams in 2D into a few archetypes, label each match by the pair (home archetype, away archetype), keep the grid cells where draws beat the market on the training fold, fit LR + polynomial, and bet draw where the model beats the implied probability by `buffer`.

`running` vs `final` = which stats the team archetypes are fit on; matches are always assigned, and the model fit, on running (point-in-time) stats. Rule #0: archetypes and cell selection are fit on the training fold only.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

CONFIG = {
    "n_team_types": 4,      # team archetypes -> n_team_types^2 grid cells (reference; swept below)
    "buffer": 0.02,
    "min_game_number": 8,
}

OUTCOME = "t_draw_flg"                                             # betting the home win
IMPLIED = {"mrkt": "mrkt_draw_impl", "b365": "b365_draw_impl"}     # implied home-win prob = 1 / home odds

# The LR bet-signal model still uses the full 4D matchup (both teams' running for/against).
RUNNING_FEATURES = [
    "homet_season_goals_for_avg", "homet_season_goals_agst_avg",
    "awayt_season_goals_for_avg", "awayt_season_goals_agst_avg",
]
# The team clustering uses one team at a time in 2D. Running (point-in-time) and final (end-of-season) forms.
HOME_RUN, AWAY_RUN = ["homet_season_goals_for_avg", "homet_season_goals_agst_avg"], ["awayt_season_goals_for_avg", "awayt_season_goals_agst_avg"]
HOME_FIN, AWAY_FIN = ["home_for_final", "home_agst_final"], ["away_for_final", "away_agst_final"]
TEAM_MAPS = {"running": (HOME_RUN, AWAY_RUN), "final": (HOME_FIN, AWAY_FIN)}

abt = pd.read_parquet("C:/Users/twluc/frame/01_data/03_abt/abt.parquet")

final_stats = pd.read_parquet("C:/Users/twluc/frame/01_data/02_features/team_season_final.parquet")
final_stats = final_stats[["team", "season", "season_goals_for_avg", "season_goals_agst_avg"]]
home_final = final_stats.rename(columns={"team": "home_team",
    "season_goals_for_avg": "home_for_final", "season_goals_agst_avg": "home_agst_final"})
away_final = final_stats.rename(columns={"team": "away_team",
    "season_goals_for_avg": "away_for_final", "season_goals_agst_avg": "away_agst_final"})
abt = abt.merge(home_final, on=["home_team", "season"], how="left").merge(away_final, on=["away_team", "season"], how="left")

matches = abt[
    (abt["homet_game_number"] > CONFIG["min_game_number"])
    & (abt["awayt_game_number"] > CONFIG["min_game_number"])
].dropna(subset=RUNNING_FEATURES + HOME_FIN + AWAY_FIN + list(IMPLIED.values()) + [OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [2]:
def pooled_roi(implied, won):
    staked = implied.sum()
    return (won.sum() - staked) / staked if staked > 0 else np.nan

def roi_grouped(bets, keys):
    totals = bets.groupby(keys).agg(won=("won", "sum"), staked=("implied", "sum"))
    return (totals["won"] - totals["staked"]) / totals["staked"]

def season_folds(data):
    for season in sorted(data["season"].unique()):
        yield ~(data["season"] == season), data["season"] == season, season

def expanding_folds(data, min_train_seasons=2):
    seasons = sorted(data["season"].unique())
    for split in range(min_train_seasons, len(seasons)):
        yield data["season"].isin(seasons[:split]), data["season"] == seasons[split], seasons[split]

def build_signal_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
                         LogisticRegression(max_iter=3000))

# Keep the grid cells where the outcome happened more often than the market priced it, on training data only.
def profitable_cells(train, cell_labels, min_matches=100):
    frame = pd.DataFrame({"cell": cell_labels, "outcome": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED["mrkt"]].to_numpy()})
    totals = frame.groupby("cell").agg(outcome=("outcome", "mean"), implied=("implied", "mean"), n=("outcome", "size"))
    return set(totals.index[(totals["n"] >= min_matches) & (totals["outcome"] > totals["implied"])])

In [3]:
# Honest per-fold run. `centroid_map` picks what the team archetypes are fit on ("running"/"final").
def collect_bets(folds, n_team_types, odds_source, centroid_map):
    home_map, away_map = TEAM_MAPS[centroid_map]
    implied_column = IMPLIED[odds_source]
    placed = []
    for train_mask, test_mask, fold_name in folds:
        train, test = matches[train_mask], matches[test_mask]

        # Team archetypes: cluster individual teams in 2D, both sides stacked, training seasons only.
        team_points = np.vstack([train[home_map].to_numpy(), train[away_map].to_numpy()])
        kmeans = KMeans(n_clusters=n_team_types, random_state=0, n_init=10).fit(team_points)

        # A match's cell = (home archetype, away archetype), always assigned by running stats.
        def cell_labels(frame):
            home_type = kmeans.predict(frame[HOME_RUN].to_numpy())
            away_type = kmeans.predict(frame[AWAY_RUN].to_numpy())
            return np.array([f"{h}-{a}" for h, a in zip(home_type, away_type)])

        train_cells = cell_labels(train)
        kept = profitable_cells(train, train_cells)
        test_cells = cell_labels(test)
        in_region_test = np.isin(test_cells, list(kept))
        if not in_region_test.any():
            continue
        in_region_train = np.isin(train_cells, list(kept))

        model = build_signal_model()
        model.fit(train[in_region_train][RUNNING_FEATURES], train[in_region_train][OUTCOME].astype(int))

        region = test[in_region_test].copy()
        region["model_prob"] = model.predict_proba(region[RUNNING_FEATURES])[:, list(model.classes_).index(1)]
        region["implied"] = region[implied_column]
        region["won"] = region[OUTCOME].astype(int)
        region["fold"] = fold_name

        is_bet = region["model_prob"] > region["implied"] + CONFIG["buffer"]
        placed.append(region.loc[is_bet, ["implied", "won", "league", "season", "fold"]])

    if not placed:
        return pd.DataFrame(columns=["implied", "won", "league", "season", "fold"])
    return pd.concat(placed, ignore_index=True)

## Gate 1 — time robustness (leave-one-season-out)

In [4]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"[{centroid_map:7s}] no bets placed under this pipeline"); continue
    per_season = roi_grouped(bets, "fold")
    positive = int((per_season > 0).sum())
    print(f"[{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          f"seasons +{positive}/{len(per_season)} {'PASS' if positive >= 3 else 'FAIL'}   ({len(bets)} bets)   "
          + "  ".join(f"{season}:{value:+.3f}" for season, value in per_season.items()))

[running] pooled +0.1014   seasons +3/4 PASS   (884 bets)   2223:+0.089  2324:+0.281  2425:-0.029  2526:+0.109


[final  ] pooled +0.1059   seasons +4/4 PASS   (1029 bets)   2223:+0.050  2324:+0.251  2425:+0.030  2526:+0.140


## Gate 2 — league robustness (survive removing the best league)

In [5]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(season_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"[{centroid_map:7s}] no bets placed under this pipeline"); continue
    without_league = {league: pooled_roi(bets[bets["league"] != league]["implied"], bets[bets["league"] != league]["won"])
                      for league in bets["league"].unique()}
    best_league = min(without_league, key=without_league.get)
    per_league = roi_grouped(bets, "league").sort_values()
    print(f"[{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          f"best league = {best_league} -> without it {without_league[best_league]:+.4f}   "
          f"GATE 2 {'PASS' if without_league[best_league] > 0 else 'FAIL'}")
    print("          per league: " + "   ".join(f"{league}:{value:+.3f}" for league, value in per_league.items()))

[running] pooled +0.1014   best league = england -> without it +0.0790   GATE 2 PASS
          per league: france:-0.021   italy:-0.016   germany:+0.122   england:+0.162   spain:+0.228


[final  ] pooled +0.1059   best league = spain -> without it +0.0826   GATE 2 PASS
          per league: italy:-0.016   france:+0.069   germany:+0.089   england:+0.140   spain:+0.269


## Directional — walk-forward and knob robustness (sweep number of team archetypes)

In [6]:
print("walk-forward (b365):")
for centroid_map in ["running", "final"]:
    bets = collect_bets(expanding_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"  [{centroid_map:7s}] no bets"); continue
    per_fold = roi_grouped(bets, "fold")
    print(f"  [{centroid_map:7s}] pooled {pooled_roi(bets['implied'], bets['won']):+.4f}   "
          + "   ".join(f"{fold}:{value:+.3f}" for fold, value in per_fold.items())
          + f"   {'VETO' if bool((per_fold < -0.03).any()) else 'no veto'}")

print("\nknob robustness — pooled ROI (b365) by number of team archetypes (cells = types^2):")
print("  types  cells  running   final")
for n_types in [2, 3, 4,5]:
    run_bets = collect_bets(season_folds(matches), n_types, "b365", "running")
    fin_bets = collect_bets(season_folds(matches), n_types, "b365", "final")
    print(f"    {n_types}     {n_types**2:2d}   {pooled_roi(run_bets['implied'], run_bets['won']):+.4f}  "
          f"{pooled_roi(fin_bets['implied'], fin_bets['won']):+.4f}")

walk-forward (b365):
  [running] pooled +0.0216   2425:-0.042   2526:+0.109   VETO


  [final  ] pooled +0.0938   2425:+0.045   2526:+0.140   no veto

knob robustness — pooled ROI (b365) by number of team archetypes (cells = types^2):
  types  cells  running   final


    2      4   +0.0504  +0.0271


    3      9   +0.0515  +0.0941


    4     16   +0.1014  +0.1059


    5     25   +0.0264  +0.0082


## Descriptive (in-sample, not a gate) — ROI per grid cell
Which type-vs-type cell drives the draw market? Team archetypes fit on all matches (running); each cell is
annotated with the two archetypes' average (goals for, goals against). In-sample, so treat as a map of where
to look, not evidence.

In [7]:
team_points = np.vstack([matches[HOME_RUN].to_numpy(), matches[AWAY_RUN].to_numpy()])
km = KMeans(n_clusters=CONFIG["n_team_types"], random_state=0, n_init=10).fit(team_points)
print("team archetypes (for, agst):")
for i, centre in enumerate(km.cluster_centers_):
    print(f"  type {i}: for={centre[0]:.2f}  agst={centre[1]:.2f}")

home_type = km.predict(matches[HOME_RUN].to_numpy())
away_type = km.predict(matches[AWAY_RUN].to_numpy())
grid = pd.DataFrame({"cell": [f"{h}-{a}" for h, a in zip(home_type, away_type)],
                     "won": matches[OUTCOME].astype(int).to_numpy(), "implied": matches["b365_draw_impl"].to_numpy()})
per_cell = grid.groupby("cell").agg(won=("won", "sum"), staked=("implied", "sum"), n=("won", "size"))
per_cell["roi"] = (per_cell["won"] - per_cell["staked"]) / per_cell["staked"]
print("\nROI per cell (home-type - away-type), in-sample b365:")
print(per_cell[["n", "roi"]].sort_values("roi", ascending=False).round(3).to_string())

team archetypes (for, agst):
  type 0: for=1.03  agst=1.30
  type 1: for=2.38  agst=0.96
  type 2: for=1.67  agst=1.24
  type 3: for=1.14  agst=1.93

ROI per cell (home-type - away-type), in-sample b365:
        n    roi
cell            
2-3   423  0.211
3-2   448  0.108
3-0   384  0.064
3-3   419  0.019
2-0   538  0.012
0-2   548 -0.070
1-2   201 -0.072
0-0   697 -0.091
0-3   375 -0.098
1-3   165 -0.130
2-1   196 -0.143
0-1   190 -0.158
3-1   163 -0.161
1-0   186 -0.205
2-2   545 -0.209
1-1    55 -0.260


## Diagnostic — where the edge lives (league × season), running map

In [8]:
for centroid_map in ["running", "final"]:
    bets = collect_bets(expanding_folds(matches), CONFIG["n_team_types"], "b365", centroid_map)
    if bets.empty:
        print(f"  [{centroid_map:7s}] no bets"); continue
    else:
        breakdown = roi_grouped(bets, ["league", "season"]).unstack("season").round(3)
        print(f"[{centroid_map:7s}] ROI by league x season (b365), total {len(bets)} bets:")
        print(breakdown.to_string())

[running] ROI by league x season (b365), total 493 bets:
season    2425   2526
league               
england  0.082  0.514
france  -0.375  0.158
germany  0.252 -0.095
italy    0.148 -0.054
spain   -0.513 -0.034
[final  ] ROI by league x season (b365), total 534 bets:
season    2425   2526
league               
england  0.022  0.220
france   0.030  0.613
germany  0.265 -0.102
italy   -0.095 -0.243
spain    0.015  0.259


## Summary

In [9]:
# ---------------------------------------------------------------------------
# SUMMARY — gates, veto, and diagnostic spread at a glance (b365, running map, default k).
# ---------------------------------------------------------------------------
k = CONFIG.get("n_clusters", CONFIG.get("n_team_types"))
bets = collect_bets(season_folds(matches), k, "b365", "running")
walk = collect_bets(expanding_folds(matches), k, "b365", "running")

if bets.empty:
    print("SUMMARY: no bets placed under this pipeline")
else:
    per_season = roi_grouped(bets, "fold")
    gate1_pass = int((per_season > 0).sum()) >= 3
    without_league = {league: pooled_roi(bets[bets["league"] != league]["implied"], bets[bets["league"] != league]["won"])
                      for league in bets["league"].unique()}
    gate2_pass = min(without_league.values()) > 0
    wf_veto = (not walk.empty) and bool((roi_grouped(walk, "fold") < -0.03).any())
    spread = roi_grouped(bets, ["league", "season"]).std()  # how noisy the per-cell breakdown is
    print(f"SUMMARY (b365, running map, k={k})")
    print(f"  pooled ROI                     {pooled_roi(bets['implied'], bets['won']):+.4f}  ({len(bets)} bets)")
    print(f"  GATE 1  (LOSO >= 3/4 seasons)    {'PASS' if gate1_pass else 'FAIL'}  (+{int((per_season > 0).sum())}/{len(per_season)})")
    print(f"  GATE 2  (drop best league)       {'PASS' if gate2_pass else 'FAIL'}  (worst {min(without_league.values()):+.4f})")
    print(f"  walk-forward                     {'VETO' if wf_veto else 'no veto'}")
    print(f"  diagnostic spread (std of league x season ROI)   {spread:.3f}")

SUMMARY (b365, running map, k=4)
  pooled ROI                     +0.1014  (884 bets)
  GATE 1  (LOSO >= 3/4 seasons)    PASS  (+3/4)
  GATE 2  (drop best league)       PASS  (worst +0.0790)
  walk-forward                     VETO
  diagnostic spread (std of league x season ROI)   0.303
